# 10. Exportação leve dos artefatos do experimento

Este notebook consolida os artefatos do experimento em um pacote compactado, mas sem duplicar arquivos grandes dentro de `exports/`.

A versão anterior da exportação copiava diretórios como `results/`, `logs/`, `data/canonical/` e `data/views/` para dentro de `exports/` antes de criar o arquivo `.zip`. Essa abordagem é simples, mas consome muito espaço, porque mantém os artefatos originais, suas cópias intermediárias e o `.zip` final ao mesmo tempo.

Neste notebook, a lógica muda para uma exportação sem cópia intermediária:

```text
1. localizar os artefatos originais no projeto;
2. criar índices e manifestos leves dentro de exports/;
3. criar o arquivo .zip lendo diretamente dos caminhos originais;
4. salvar o .zip fora de exports/, em uma pasta separada de zip files.
```

Com isso, a pasta `exports/` passa a armazenar apenas metadados pequenos, como `README.md`, `export_manifest.json`, índices de arquivos e registros de fontes ausentes ou ignoradas. Os arquivos reais continuam nos seus locais originais e são adicionados diretamente ao `.zip`.

O resultado final será salvo em:

```text
/workspace/zip_files/
```

Essa organização evita duplicação desnecessária e reduz o risco de falta de espaço durante a exportação.


## 1. Escopo da exportação

A exportação deste notebook tem o objetivo de reunir praticamente todos os artefatos úteis para auditoria, revisão, reprodução parcial e escrita do relatório final.

Entram na exportação, quando existirem:

```text
configs/
manifests/
logs/
data/canonical/
data/views/
results/
notebooks/
scripts/
requirements*.txt
README.md
pyproject.toml
```

Também entram os arquivos leves gerados pelo próprio notebook dentro de `exports/`, como:

```text
README.md
export_manifest.json
export_manifest.md
export_summary.json
index/file_index.csv
index/file_index.json
index/missing_optional_sources.json
index/skipped_files.json
archive_summary.json
```

Por padrão, não entram artefatos muito grandes ou pouco portáveis, como:

```text
.venv/
data/cache/
adapters/
outputs/adapters/
cache do Hugging Face
__pycache__/
.ipynb_checkpoints/
```

Os adaptadores LoRA/QLoRA podem ser incluídos explicitamente alterando uma flag de configuração. O padrão é não exportá-los, porque eles podem aumentar bastante o tamanho do pacote final.


## 2. Imports e caminhos principais

Esta etapa define os imports usados pelo notebook e os caminhos principais do projeto.

A raiz esperada do projeto é:

```text
/workspace/pi-defense-exp
```

Os metadados leves da exportação serão criados dentro de:

```text
/workspace/pi-defense-exp/exports/experiment_artifacts/full/
```

O arquivo compactado final será salvo fora da pasta `exports/`, em:

```text
/workspace/zip_files/
```

Essa separação é importante porque evita colocar o `.zip` dentro da própria árvore que poderia ser compactada.


In [1]:
import hashlib
import json
import math
import time
import zipfile
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import pandas as pd


In [2]:
PROJECT_ROOT = Path("/workspace/pi-defense-exp")
RUN_MODE = "full"

EXPORT_ROOT = PROJECT_ROOT / "exports" / "experiment_artifacts" / RUN_MODE
EXPORT_INDEX_DIR = EXPORT_ROOT / "index"

ZIP_FILES_DIR = PROJECT_ROOT.parent / "zip_files"
ZIP_FILES_DIR.mkdir(parents=True, exist_ok=True)

EXPORT_ROOT.mkdir(parents=True, exist_ok=True)
EXPORT_INDEX_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Export metadata root:", EXPORT_ROOT)
print("Zip files dir:", ZIP_FILES_DIR)


Project root: /workspace/pi-defense-exp
Export metadata root: /workspace/pi-defense-exp/exports/experiment_artifacts/full
Zip files dir: /workspace/zip_files


## 3. Configurações de inclusão e exclusão

Esta célula controla quais grupos de artefatos entram no pacote final.

A exportação foi desenhada para incluir quase tudo que é necessário para revisar o experimento, mas evitando copiar arquivos grandes de ambiente, cache ou pesos de modelo por padrão.

A flag mais sensível é `EXPORT_MODEL_ADAPTERS`. Quando ela está como `False`, os adaptadores treinados ficam fora do `.zip`. Isso é proposital para economizar espaço. Caso seja necessário arquivar também os adaptadores LoRA/QLoRA, altere para `True` antes de executar o notebook.


In [13]:
EXPORT_CONFIGS = True
EXPORT_MANIFESTS = True
EXPORT_LOGS = True
EXPORT_DATA_CANONICAL = True
EXPORT_DATA_VIEWS = True
EXPORT_RESULTS = True
EXPORT_NOTEBOOKS = True
EXPORT_SCRIPTS = True
EXPORT_REQUIREMENTS_AND_ROOT_FILES = True

EXPORT_MODEL_ADAPTERS = True
EXPORT_DATA_CACHE = False
EXPORT_ENVIRONMENT = True

COMPRESSION_LEVEL = 6
ARCHIVE_NAME = f"experiment_artifacts_{RUN_MODE}.zip"
ARCHIVE_PATH = ZIP_FILES_DIR / ARCHIVE_NAME
ZIP_INFO_PATH = ZIP_FILES_DIR / f"experiment_artifacts_{RUN_MODE}_zip_info.json"

EXPORT_POLICY = {
    "export_mode": "zip_from_sources_no_intermediate_copy",
    "run_mode": RUN_MODE,
    "export_root": str(EXPORT_ROOT),
    "zip_files_dir": str(ZIP_FILES_DIR),
    "archive_path": str(ARCHIVE_PATH),
    "include_configs": EXPORT_CONFIGS,
    "include_manifests": EXPORT_MANIFESTS,
    "include_logs": EXPORT_LOGS,
    "include_data_canonical": EXPORT_DATA_CANONICAL,
    "include_data_views": EXPORT_DATA_VIEWS,
    "include_results": EXPORT_RESULTS,
    "include_notebooks": EXPORT_NOTEBOOKS,
    "include_scripts": EXPORT_SCRIPTS,
    "include_model_adapters": EXPORT_MODEL_ADAPTERS,
    "include_data_cache": EXPORT_DATA_CACHE,
    "include_environment": EXPORT_ENVIRONMENT,
}

EXPORT_POLICY


{'export_mode': 'zip_from_sources_no_intermediate_copy',
 'run_mode': 'full',
 'export_root': '/workspace/pi-defense-exp/exports/experiment_artifacts/full',
 'zip_files_dir': '/workspace/zip_files',
 'archive_path': '/workspace/zip_files/experiment_artifacts_full.zip',
 'include_configs': True,
 'include_manifests': True,
 'include_logs': True,
 'include_data_canonical': True,
 'include_data_views': True,
 'include_results': True,
 'include_notebooks': True,
 'include_scripts': True,
 'include_model_adapters': True,
 'include_data_cache': False,
 'include_environment': True}

## 4. Funções utilitárias

As funções desta seção fazem três coisas principais:

```text
1. salvar JSON de forma segura;
2. criar índices de arquivos;
3. calcular tamanho e hash do arquivo compactado.
```

A função de sanitização evita salvar valores como `NaN`, `Infinity` ou `-Infinity` em JSON inválido. Quando esses valores aparecem, eles são convertidos para strings.


In [4]:
def utc_now() -> str:
    return datetime.now(timezone.utc).isoformat()


def sanitize_json_value(value: Any) -> Any:
    if isinstance(value, dict):
        return {str(k): sanitize_json_value(v) for k, v in value.items()}
    if isinstance(value, list):
        return [sanitize_json_value(v) for v in value]
    if isinstance(value, tuple):
        return [sanitize_json_value(v) for v in value]
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, float):
        if math.isnan(value):
            return "NaN"
        if math.isinf(value):
            return "Infinity" if value > 0 else "-Infinity"
    return value


def write_json(path: Path, data: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    sanitized = sanitize_json_value(data)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(sanitized, f, indent=2, ensure_ascii=False, allow_nan=False)


def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def file_size_mb(path: Path) -> float:
    return path.stat().st_size / (1024 * 1024)


## 5. Política de exclusão

A exportação deve incluir os artefatos relevantes, mas não deve incluir automaticamente caches, ambiente virtual, checkpoints automáticos do Jupyter ou pesos/adaptadores quando a flag correspondente estiver desativada.

Arquivos excluídos por essa política aparecem em:

```text
index/skipped_files.json
```

Isso é diferente de uma fonte opcional ausente. Quando uma pasta esperada não existe, ela aparece em:

```text
index/missing_optional_sources.json
```

A separação ajuda a distinguir dois casos:

```text
source_missing = a fonte não existia
skipped_by_policy = a fonte existia, mas foi ignorada de propósito
```


In [5]:
EXCLUDED_DIR_NAMES = {
    ".git",
    ".venv",
    "__pycache__",
    ".ipynb_checkpoints",
}

EXCLUDED_FILE_SUFFIXES = {
    ".pyc",
    ".pyo",
    ".tmp",
    ".lock",
}

skipped_files: list[dict] = []
missing_optional_sources: list[dict] = []


def relative_to_project(path: Path) -> str:
    try:
        return path.resolve().relative_to(PROJECT_ROOT.resolve()).as_posix()
    except ValueError:
        return path.resolve().as_posix()


def archive_path_for_project_file(path: Path) -> str:
    return f"experiment_artifacts/{RUN_MODE}/{relative_to_project(path)}"


def should_skip_path(path: Path) -> tuple[bool, str | None]:
    path = path.resolve()

    if any(part in EXCLUDED_DIR_NAMES for part in path.parts):
        return True, "excluded_directory_name"

    if path.suffix in EXCLUDED_FILE_SUFFIXES:
        return True, "excluded_file_suffix"

    rel = relative_to_project(path)

    if rel.startswith("data/cache/") and not EXPORT_DATA_CACHE:
        return True, "data_cache_disabled"
    if rel == "data/cache" and not EXPORT_DATA_CACHE:
        return True, "data_cache_disabled"

    if rel.startswith("adapters/") and not EXPORT_MODEL_ADAPTERS:
        return True, "model_adapters_disabled"
    if rel == "adapters" and not EXPORT_MODEL_ADAPTERS:
        return True, "model_adapters_disabled"

    if rel.startswith("outputs/adapters/") and not EXPORT_MODEL_ADAPTERS:
        return True, "model_adapters_disabled"
    if rel == "outputs/adapters" and not EXPORT_MODEL_ADAPTERS:
        return True, "model_adapters_disabled"

    if rel.startswith("exports/archives/"):
        return True, "old_archives_not_included"

    return False, None


def register_skipped(path: Path, group: str, reason: str) -> None:
    skipped_files.append({
        "group": group,
        "path": str(path),
        "project_relative_path": relative_to_project(path),
        "reason": reason,
    })


def register_missing(path: Path, group: str, reason: str = "source_missing") -> None:
    missing_optional_sources.append({
        "group": group,
        "path": str(path),
        "reason": reason,
    })


## 6. Plano de fontes da exportação

Esta seção define quais pastas e arquivos serão considerados para o `.zip`.

O notebook não copia essas fontes para `exports/`. Ele apenas cria um índice com os caminhos originais e depois adiciona esses arquivos diretamente ao arquivo compactado.

Essa é a parte central da nova lógica de economia de espaço.


In [6]:
source_dirs: list[dict] = []
source_files: list[dict] = []


def add_source_dir(group: str, path: Path, enabled: bool = True, required: bool = False) -> None:
    if not enabled:
        if path.exists():
            register_skipped(path, group, "disabled_by_configuration")
        return
    if path.exists() and path.is_dir():
        source_dirs.append({"group": group, "path": path, "required": required})
    else:
        register_missing(path, group, "required_source_missing" if required else "source_missing")


def add_source_file(group: str, path: Path, enabled: bool = True, required: bool = False) -> None:
    if not enabled:
        if path.exists():
            register_skipped(path, group, "disabled_by_configuration")
        return
    if path.exists() and path.is_file():
        source_files.append({"group": group, "path": path, "required": required})
    else:
        register_missing(path, group, "required_source_missing" if required else "source_missing")


add_source_dir("configs", PROJECT_ROOT / "configs", EXPORT_CONFIGS)
add_source_dir("manifests", PROJECT_ROOT / "manifests", EXPORT_MANIFESTS)
add_source_dir("logs", PROJECT_ROOT / "logs", EXPORT_LOGS)
add_source_dir("data_canonical", PROJECT_ROOT / "data" / "canonical", EXPORT_DATA_CANONICAL)
add_source_dir("data_views", PROJECT_ROOT / "data" / "views", EXPORT_DATA_VIEWS)
add_source_dir("results", PROJECT_ROOT / "results", EXPORT_RESULTS)
add_source_dir("scripts", PROJECT_ROOT / "scripts", EXPORT_SCRIPTS)
add_source_dir("notebooks_directory", PROJECT_ROOT / "notebooks", EXPORT_NOTEBOOKS)

add_source_dir("data_cache", PROJECT_ROOT / "data" / "cache", EXPORT_DATA_CACHE)
add_source_dir("model_adapters", PROJECT_ROOT / "adapters", EXPORT_MODEL_ADAPTERS)
add_source_dir("model_adapters_outputs", PROJECT_ROOT / "outputs" / "adapters", EXPORT_MODEL_ADAPTERS)

if EXPORT_REQUIREMENTS_AND_ROOT_FILES:
    for filename in [
        "README.md",
        "requirements-data.txt",
        "requirements-train.txt",
        "requirements.txt",
        "pyproject.toml",
        ".env.example",
    ]:
        add_source_file("root_files", PROJECT_ROOT / filename, enabled=True, required=False)

if EXPORT_NOTEBOOKS:
    for notebook_path in sorted(PROJECT_ROOT.glob("*.ipynb")):
        add_source_file("notebooks_root", notebook_path, enabled=True, required=False)

print("Source directories:")
for source in source_dirs:
    print("-", source["group"], source["path"])

print("\nSource files:")
for source in source_files:
    print("-", source["group"], source["path"])

print("\nMissing optional sources:", len(missing_optional_sources))
print("Skipped by policy/configuration:", len(skipped_files))


Source directories:
- configs /workspace/pi-defense-exp/configs
- manifests /workspace/pi-defense-exp/manifests
- logs /workspace/pi-defense-exp/logs
- data_canonical /workspace/pi-defense-exp/data/canonical
- data_views /workspace/pi-defense-exp/data/views
- results /workspace/pi-defense-exp/results
- scripts /workspace/pi-defense-exp/scripts
- notebooks_directory /workspace/pi-defense-exp/notebooks

Source files:
- root_files /workspace/pi-defense-exp/requirements-data.txt
- root_files /workspace/pi-defense-exp/requirements-train.txt

Missing optional sources: 4
Skipped by policy/configuration: 2


## 7. Indexar arquivos sem copiar

Esta etapa percorre as fontes configuradas e monta uma lista de arquivos que entrarão no `.zip`.

Cada item do índice possui:

```text
group
source_path
project_relative_path
archive_path
size_bytes
size_mb
```

O campo `archive_path` indica o caminho interno que o arquivo terá dentro do `.zip`. Isso permite que o pacote final mantenha uma estrutura organizada sem precisar copiar os arquivos antes.


In [7]:
file_entries: list[dict] = []
seen_source_paths: set[str] = set()


def add_file_entry(path: Path, group: str) -> None:
    path = path.resolve()
    source_key = str(path)
    if source_key in seen_source_paths:
        return

    should_skip, reason = should_skip_path(path)
    if should_skip:
        register_skipped(path, group, reason or "skipped")
        return

    seen_source_paths.add(source_key)
    stat = path.stat()
    file_entries.append({
        "group": group,
        "source_path": str(path),
        "project_relative_path": relative_to_project(path),
        "archive_path": archive_path_for_project_file(path),
        "size_bytes": stat.st_size,
        "size_mb": stat.st_size / (1024 * 1024),
        "modified_at_utc": datetime.fromtimestamp(stat.st_mtime, tz=timezone.utc).isoformat(),
    })


for source in source_files:
    add_file_entry(source["path"], source["group"])

for source in source_dirs:
    root = source["path"]
    group = source["group"]

    should_skip_root, reason = should_skip_path(root)
    if should_skip_root:
        register_skipped(root, group, reason or "skipped")
        continue

    for path in sorted(root.rglob("*")):
        if path.is_file():
            add_file_entry(path, group)

file_index_df = pd.DataFrame(file_entries)

print("Arquivos indexados para exportação:", len(file_index_df))
if not file_index_df.empty:
    print("Tamanho total estimado dos arquivos indexados:", f"{file_index_df['size_mb'].sum():.2f} MB")
    display(file_index_df.groupby("group").agg(files=("source_path", "count"), size_mb=("size_mb", "sum")).reset_index())
else:
    display(file_index_df)


Arquivos indexados para exportação: 342
Tamanho total estimado dos arquivos indexados: 696.67 MB


,group,files,size_mb
0,configs,2,0.009270
1,data_canonical,7,15.923454
2,data_views,9,20.715639
3,logs,99,2.448759
4,manifests,20,0.162924
5,notebooks_directory,10,2.014692
6,results,193,655.397265
7,root_files,2,0.000179


## 8. Criar metadados leves em `exports/`

Agora o notebook gera os arquivos leves da exportação.

Esses arquivos ficam fisicamente dentro de `exports/`, mas são pequenos. Eles servem para explicar o conteúdo do pacote, registrar a política de inclusão/exclusão, listar arquivos indexados e documentar fontes opcionais ausentes.

Esses metadados também serão incluídos no `.zip` final.


In [8]:
export_started_at = utc_now()

summary_by_group = []
if not file_index_df.empty:
    grouped = file_index_df.groupby("group").agg(
        files=("source_path", "count"),
        size_mb=("size_mb", "sum"),
    ).reset_index()
    summary_by_group = grouped.to_dict(orient="records")

export_summary = {
    "notebook": "10_export_experiment_artifacts",
    "created_at_utc": export_started_at,
    "project_root": str(PROJECT_ROOT),
    "run_mode": RUN_MODE,
    "export_mode": "zip_from_sources_no_intermediate_copy",
    "export_root": str(EXPORT_ROOT),
    "zip_files_dir": str(ZIP_FILES_DIR),
    "archive_path": str(ARCHIVE_PATH),
    "total_indexed_files": int(len(file_entries)),
    "total_indexed_size_mb": float(file_index_df["size_mb"].sum()) if not file_index_df.empty else 0.0,
    "summary_by_group": summary_by_group,
    "missing_optional_sources_count": len(missing_optional_sources),
    "skipped_files_count": len(skipped_files),
    "export_policy": EXPORT_POLICY,
}

readme_md = f"""# Exportação dos artefatos do experimento

Esta pasta contém os metadados leves da exportação do experimento.

A exportação foi feita no modo:

```text
zip_from_sources_no_intermediate_copy
```

Nesse modo, os artefatos originais não são copiados para dentro de `exports/`. O notebook apenas gera índices e manifestos leves aqui, e o arquivo `.zip` final é criado lendo diretamente os arquivos originais do projeto.

## Caminhos principais

- Projeto: `{PROJECT_ROOT}`
- Metadados da exportação: `{EXPORT_ROOT}`
- Arquivo compactado final: `{ARCHIVE_PATH}`

## Arquivos importantes

- `export_summary.json`: resumo geral da exportação.
- `export_manifest.json`: manifesto estruturado da exportação.
- `export_manifest.md`: manifesto legível em Markdown.
- `index/file_index.csv`: índice tabular dos arquivos incluídos no `.zip`.
- `index/file_index.json`: índice em JSON dos arquivos incluídos no `.zip`.
- `index/missing_optional_sources.json`: fontes opcionais que não existiam no projeto.
- `index/skipped_files.json`: arquivos ou pastas ignorados por política de exportação.

## Observação sobre espaço em disco

Como os arquivos originais não são copiados para `exports/`, esta pasta deve permanecer leve. O arquivo `.zip` final fica em uma pasta separada chamada `zip_files`.
"""

manifest_md = f"""# Manifesto da exportação

## Identificação

- Notebook: `10_export_experiment_artifacts`
- Criado em UTC: `{export_started_at}`
- Projeto: `{PROJECT_ROOT}`
- Modo da execução: `{RUN_MODE}`
- Modo da exportação: `zip_from_sources_no_intermediate_copy`

## Escopo

A exportação indexa os artefatos originais do projeto e cria um `.zip` diretamente a partir desses caminhos, sem cópia intermediária dos diretórios grandes para `exports/`.

## Totais planejados

- Arquivos indexados: `{len(file_entries)}`
- Tamanho total estimado: `{export_summary['total_indexed_size_mb']:.2f} MB`
- Fontes opcionais ausentes: `{len(missing_optional_sources)}`
- Itens ignorados por política: `{len(skipped_files)}`

## Arquivo compactado final

O arquivo compactado será gerado em:

```text
{ARCHIVE_PATH}
```

## Itens excluídos por padrão

Por padrão, não são incluídos caches, ambiente virtual, checkpoints automáticos, `__pycache__` e adaptadores de modelo, a menos que a configuração correspondente seja habilitada.
"""

write_json(EXPORT_ROOT / "export_summary.json", export_summary)
write_json(EXPORT_ROOT / "export_manifest.json", {
    "notebook": "10_export_experiment_artifacts",
    "created_at_utc": export_started_at,
    "project_root": str(PROJECT_ROOT),
    "run_mode": RUN_MODE,
    "export_mode": "zip_from_sources_no_intermediate_copy",
    "export_policy": EXPORT_POLICY,
    "summary": export_summary,
})

(EXPORT_ROOT / "README.md").write_text(readme_md, encoding="utf-8")
(EXPORT_ROOT / "export_manifest.md").write_text(manifest_md, encoding="utf-8")

write_json(EXPORT_INDEX_DIR / "missing_optional_sources.json", missing_optional_sources)
write_json(EXPORT_INDEX_DIR / "skipped_files.json", skipped_files)

print("Metadados leves criados em:", EXPORT_ROOT)


Metadados leves criados em: /workspace/pi-defense-exp/exports/experiment_artifacts/full


## 9. Adicionar os metadados gerados ao índice

Depois de criar os metadados em `exports/`, eles também precisam entrar no `.zip`.

Nesta etapa, o notebook adiciona ao índice os arquivos leves que acabou de gerar.


In [9]:
generated_metadata_files = [
    EXPORT_ROOT / "README.md",
    EXPORT_ROOT / "export_summary.json",
    EXPORT_ROOT / "export_manifest.json",
    EXPORT_ROOT / "export_manifest.md",
    EXPORT_INDEX_DIR / "missing_optional_sources.json",
    EXPORT_INDEX_DIR / "skipped_files.json",
]

for metadata_path in generated_metadata_files:
    if metadata_path.exists():
        add_file_entry(metadata_path, "export_metadata")

file_index_df = pd.DataFrame(file_entries)
file_index_csv_path = EXPORT_INDEX_DIR / "file_index.csv"
file_index_json_path = EXPORT_INDEX_DIR / "file_index.json"

file_index_df.to_csv(file_index_csv_path, index=False)
write_json(file_index_json_path, file_index_df.to_dict(orient="records"))

for index_path in [file_index_csv_path, file_index_json_path]:
    if index_path.exists():
        add_file_entry(index_path, "export_metadata")

file_index_df = pd.DataFrame(file_entries)
file_index_df.to_csv(file_index_csv_path, index=False)
write_json(file_index_json_path, file_index_df.to_dict(orient="records"))

print("Índice final de arquivos:", file_index_csv_path)
print("Arquivos totais no índice final:", len(file_index_df))
display(file_index_df.groupby("group").agg(files=("source_path", "count"), size_mb=("size_mb", "sum")).reset_index())


Índice final de arquivos: /workspace/pi-defense-exp/exports/experiment_artifacts/full/index/file_index.csv
Arquivos totais no índice final: 350


,group,files,size_mb
0,configs,2,0.009270
1,data_canonical,7,15.923454
2,data_views,9,20.715639
3,export_metadata,8,0.276744
4,logs,99,2.448759
5,manifests,20,0.162924
6,notebooks_directory,10,2.014692
7,results,193,655.397265
8,root_files,2,0.000179


## 10. Criar o `.zip` diretamente a partir dos arquivos originais

Esta etapa cria o arquivo compactado final.

A diferença importante é que os arquivos são adicionados ao `.zip` diretamente a partir de seus caminhos originais. O notebook não copia `results/`, `logs/`, `data/` ou outros diretórios grandes para `exports/` antes de compactar.

O arquivo final será salvo em:

```text
/workspace/zip_files/
```


In [10]:
if file_index_df.empty:
    raise RuntimeError("Nenhum arquivo foi indexado para exportação. O .zip não será criado.")

if ARCHIVE_PATH.exists():
    ARCHIVE_PATH.unlink()

archive_started_at = utc_now()
start_time = time.time()

with zipfile.ZipFile(
    ARCHIVE_PATH,
    mode="w",
    compression=zipfile.ZIP_DEFLATED,
    compresslevel=COMPRESSION_LEVEL,
) as zip_file:
    for row in file_index_df.to_dict(orient="records"):
        source_path = Path(row["source_path"])
        archive_path = row["archive_path"]

        if not source_path.exists():
            raise FileNotFoundError(
                f"Arquivo indexado não encontrado durante compactação: {source_path}"
            )

        zip_file.write(source_path, archive_path)

archive_elapsed_seconds = time.time() - start_time
archive_finished_at = utc_now()

archive_info = {
    "archive_path": str(ARCHIVE_PATH),
    "archive_name": ARCHIVE_NAME,
    "created_at_utc": archive_finished_at,
    "started_at_utc": archive_started_at,
    "elapsed_seconds": archive_elapsed_seconds,
    "compression_level": COMPRESSION_LEVEL,
    "files_in_archive": int(len(file_index_df)),
    "source_total_size_mb": float(file_index_df["size_mb"].sum()),
    "archive_size_mb": file_size_mb(ARCHIVE_PATH),
    "sha256": sha256_file(ARCHIVE_PATH),
}

write_json(ZIP_INFO_PATH, archive_info)
write_json(EXPORT_ROOT / "archive_summary.json", archive_info)

print("Arquivo compactado criado:", ARCHIVE_PATH)
print("Arquivos incluídos:", archive_info["files_in_archive"])
print("Tamanho do zip:", f"{archive_info['archive_size_mb']:.2f} MB")
print("SHA256:", archive_info["sha256"])


Arquivo compactado criado: /workspace/zip_files/experiment_artifacts_full.zip
Arquivos incluídos: 350
Tamanho do zip: 12.90 MB
SHA256: 7d61e143c722a03ede8c44b72e96ed16472306005cea8d3db2ac086d8b6aaab8


## 11. Conferência final

Esta etapa verifica se os principais arquivos da exportação foram criados.

A conferência separa:

```text
metadados leves em exports/
arquivo compactado em zip_files/
```

Essa separação confirma que a exportação não dependeu de uma cópia intermediária dos diretórios grandes.


In [11]:
expected_outputs = {
    "export_readme": EXPORT_ROOT / "README.md",
    "export_summary_json": EXPORT_ROOT / "export_summary.json",
    "export_manifest_json": EXPORT_ROOT / "export_manifest.json",
    "export_manifest_md": EXPORT_ROOT / "export_manifest.md",
    "file_index_csv": EXPORT_INDEX_DIR / "file_index.csv",
    "file_index_json": EXPORT_INDEX_DIR / "file_index.json",
    "missing_optional_sources_json": EXPORT_INDEX_DIR / "missing_optional_sources.json",
    "skipped_files_json": EXPORT_INDEX_DIR / "skipped_files.json",
    "archive_summary_json": EXPORT_ROOT / "archive_summary.json",
    "zip_archive": ARCHIVE_PATH,
    "zip_info_json": ZIP_INFO_PATH,
}

final_check_rows = []
for name, path in expected_outputs.items():
    final_check_rows.append({
        "name": name,
        "path": str(path),
        "exists": path.exists(),
        "size_mb": file_size_mb(path) if path.exists() and path.is_file() else None,
    })

final_check_df = pd.DataFrame(final_check_rows)
display(final_check_df)

if not final_check_df["exists"].all():
    missing = final_check_df.loc[~final_check_df["exists"], "name"].tolist()
    raise RuntimeError("Arquivos esperados não foram criados: " + ", ".join(missing))

print("Conferência final concluída com sucesso.")


,name,path,exists,size_mb
0,export_readme,/workspace/pi-defense-exp/exports/experiment_a...,True,0.001341
1,export_summary_json,/workspace/pi-defense-exp/exports/experiment_a...,True,0.001938
2,export_manifest_json,/workspace/pi-defense-exp/exports/experiment_a...,True,0.002940
3,export_manifest_md,/workspace/pi-defense-exp/exports/experiment_a...,True,0.000935
4,file_index_csv,/workspace/pi-defense-exp/exports/experiment_a...,True,0.105039
5,file_index_json,/workspace/pi-defense-exp/exports/experiment_a...,True,0.159030
6,missing_optional_sources_json,/workspace/pi-defense-exp/exports/experiment_a...,True,0.000466
7,skipped_files_json,/workspace/pi-defense-exp/exports/experiment_a...,True,0.006486
8,archive_summary_json,/workspace/pi-defense-exp/exports/experiment_a...,True,0.000469
9,zip_archive,/workspace/zip_files/experiment_artifacts_full...,True,12.898014


Conferência final concluída com sucesso.


## 12. Como interpretar `missing_optional_sources` e `skipped_files`

A exportação gera dois arquivos de auditoria diferentes:

```text
index/missing_optional_sources.json
index/skipped_files.json
```

O primeiro registra fontes opcionais que o notebook procurou, mas que não existiam no projeto. Isso normalmente não indica erro. Por exemplo, se um notebook anterior não gerou figuras, a pasta de figuras pode aparecer como ausente.

O segundo registra itens que existiam ou poderiam existir, mas foram ignorados por política de exportação. Exemplos comuns são caches, ambiente virtual, checkpoints automáticos e adaptadores quando `EXPORT_MODEL_ADAPTERS=False`.

Essa distinção ajuda a revisar a exportação sem confundir ausência normal de uma pasta opcional com exclusão deliberada de arquivos grandes.


In [12]:
print("Missing optional sources:", len(missing_optional_sources))
if missing_optional_sources:
    display(pd.DataFrame(missing_optional_sources))
else:
    print("Nenhuma fonte opcional ausente registrada.")

print("\nSkipped files/directories:", len(skipped_files))
if skipped_files:
    display(pd.DataFrame(skipped_files).head(50))
else:
    print("Nenhum item ignorado por política registrado.")


Missing optional sources: 4


,group,path,reason
0,root_files,/workspace/pi-defense-exp/README.md,source_missing
1,root_files,/workspace/pi-defense-exp/requirements.txt,source_missing
2,root_files,/workspace/pi-defense-exp/pyproject.toml,source_missing
3,root_files,/workspace/pi-defense-exp/.env.example,source_missing



Skipped files/directories: 23


,group,path,project_relative_path,reason
0,data_cache,/workspace/pi-defense-exp/data/cache,data/cache,disabled_by_configuration
1,model_adapters,/workspace/pi-defense-exp/adapters,adapters,disabled_by_configuration
2,logs,/workspace/pi-defense-exp/logs/metrics/.ipynb_...,logs/metrics/.ipynb_checkpoints/06_compute_met...,excluded_directory_name
3,logs,/workspace/pi-defense-exp/logs/training/.ipynb...,logs/training/.ipynb_checkpoints/04_run_traini...,excluded_directory_name
4,logs,/workspace/pi-defense-exp/logs/training/.ipynb...,logs/training/.ipynb_checkpoints/04_run_traini...,excluded_directory_name
5,logs,/workspace/pi-defense-exp/logs/training/.ipynb...,logs/training/.ipynb_checkpoints/04_run_traini...,excluded_directory_name
6,logs,/workspace/pi-defense-exp/logs/training/runs/c...,logs/training/runs/c3_secalign_dpo/seed_42/.ip...,excluded_directory_name
7,logs,/workspace/pi-defense-exp/logs/training/runs/c...,logs/training/runs/c4_ih_sft/seed_123/.ipynb_c...,excluded_directory_name
8,logs,/workspace/pi-defense-exp/logs/training/runs/c...,logs/training/runs/c4_ih_sft/seed_42/.ipynb_ch...,excluded_directory_name
9,results,/workspace/pi-defense-exp/results/inference/fu...,results/inference/full/c0_base/seed_42/.ipynb_...,excluded_directory_name


## 13. Próximos passos

O pacote compactado criado por este notebook pode ser usado para arquivar ou transferir os artefatos do experimento.

O arquivo principal é:

```text
/workspace/zip_files/experiment_artifacts_full.zip
```

O arquivo com informações do `.zip`, incluindo tamanho e hash SHA256, fica em:

```text
/workspace/zip_files/experiment_artifacts_full_zip_info.json
```

Como esta exportação não copia os artefatos grandes para `exports/`, a pasta `exports/experiment_artifacts/full/` deve permanecer relativamente pequena. Ela contém apenas metadados, índices e manifestos da exportação.

Caso seja necessário incluir os adaptadores treinados no pacote final, altere:

```python
EXPORT_MODEL_ADAPTERS = True
```

e execute novamente o notebook. Essa opção pode aumentar significativamente o tamanho do arquivo `.zip`.
